In [33]:
import os
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# Define the path to the main directory containing subdirectories for each class
data_directory = 'data/'

In [34]:
# Specify the image dimensions and batch size
img_width, img_height = 128, 128
batch_size = 100
# Use ImageDataGenerator for data augmentation and normalization
datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # Adjust the validation split ratio as needed
)

In [35]:
# Create separate generators for training and validation sets
train_generator = datagen.flow_from_directory(
    "data/training",
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    classes=['Unidentified', 'Hyalomma', 'Rhipicephalus'],
    subset='training'  # Specify 'training' for the training set
)

validation_generator = datagen.flow_from_directory(
    "data/validation",
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    classes=['Unidentified', 'Hyalomma', 'Rhipicephalus'],
    subset='validation'  # Specify 'validation' for the validation set
)

Found 3449 images belonging to 3 classes.
Found 429 images belonging to 3 classes.


In [37]:
# Build a simple CNN model
model = Sequential()
model.add(Conv2D(100, (3, 3), input_shape=(img_width, img_height, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(3, activation='softmax'))  # Assuming you have 3 classes

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [38]:
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    epochs=5,  # Adjust the number of epochs as needed
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size
)

Epoch 1/5
34/34 [==============================] - 111s 3s/step - loss: 4.2988 - accuracy: 0.4778 - val_loss: 0.5936 - val_accuracy: 0.7175
Epoch 2/5
34/34 [==============================] - 109s 3s/step - loss: 0.3287 - accuracy: 0.8752 - val_loss: 0.4141 - val_accuracy: 0.8250
Epoch 3/5
34/34 [==============================] - 102s 3s/step - loss: 0.1594 - accuracy: 0.9457 - val_loss: 0.2549 - val_accuracy: 0.8675
Epoch 4/5
34/34 [==============================] - 111s 3s/step - loss: 0.1461 - accuracy: 0.9448 - val_loss: 0.2234 - val_accuracy: 0.9200
Epoch 5/5
34/34 [==============================] - 107s 3s/step - loss: 0.1092 - accuracy: 0.9645 - val_loss: 0.2083 - val_accuracy: 0.9175


In [40]:
# Evaluate the model on the test set (you should have a separate test set)
test_generator = datagen.flow_from_directory(
    "data/testing",
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    classes=['Unidentified', 'Hyalomma', 'Rhipicephalus'],
    subset='validation'
)

Found 143 images belonging to 3 classes.


In [42]:
import numpy as np

# Make predictions on the test set
test_loss, test_accuracy = model.evaluate(test_generator, steps = test_generator.samples // batch_size)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

y_true = test_generator.classes
y_pred_probs = model.predict(test_generator, steps = None)

print("test_generator.samples:", test_generator.samples)
print("len(y_true):", len(y_true))

y_pred = np.argmax(y_pred_probs, axis=1)
print("y_pred: ", len(y_pred));

# # Calculate confusion matrix
# conf_matrix = confusion_matrix(y_true, y_pred)

# Calculate accuracy
accuracy = accuracy_score(y_true, y_pred)

# Calculate precision
precision = precision_score(y_true, y_pred, average='weighted')

# Calculate sensitivity (recall)
sensitivity = recall_score(y_true, y_pred, average='weighted')

# Print the results
# print("Confusion Matrix:")
# print(conf_matrix)
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Sensitivity (Recall):", sensitivity)

1/1 [==============================] - 2s 2s/step - loss: 0.1973 - accuracy: 0.9200
Test Loss: 0.19733814895153046
Test Accuracy: 0.9200000166893005
2/2 [==============================] - 3s 751ms/step
test_generator.samples: 143
len(y_true): 143
y_pred:  143
Accuracy: 0.42657342657342656
Precision: 0.4294587526294844
Sensitivity (Recall): 0.42657342657342656


In [43]:
model.save('tickBugsModelV3.h5')

/opt/homebrew/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
